# Model 2: Seq2Seq with Luong Attention (Luong Dot Attention)

**Purpose:**  
Implement a sequence-to-sequence LSTM model **with Luong Attention** for the WikiQA chatbot.  
This is the second model for the group project.

**What this notebook does:**
1. Loads pre-processed data from `correct-answers-only`
2. Builds Encoder + Luong Attention Decoder
3. Trains the model
4. Evaluates with BLEU score + sample outputs
5. Compares with Model 1 (no attention)


## Section 1: Imports and Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
from pathlib import Path
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import random

PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

EMBED_DIM = 256
HIDDEN_DIM = 256
N_LAYERS = 1
DROPOUT = 0.5
BATCH_SIZE = 32
TEACHER_FORCING_RATIO = 0.5
LEARNING_RATE = 5e-4
CLIP = 1.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Section 2: Encoder

Same Encoder as Model 1. Converts the input question into hidden states.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers, 
                           batch_first=True, 
                           dropout=dropout if n_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_lens):
        embedded = self.dropout(self.embedding(src))
        packed = nn.utils.rnn.pack_padded_sequence(embedded, src_lens.cpu(), 
                                                  batch_first=True, enforce_sorted=False)
        packed_output, hidden = self.lstm(packed)
        encoder_outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)
        return encoder_outputs, hidden

## Section 3: Luong Attention Decoder (Model 2 Core)

At every decoding step, it uses **Luong Dot Attention** to focus on the most relevant words in the question.

In [ ]:
class LuongAttnDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers, 
                           batch_first=True, 
                           dropout=dropout if n_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        
        self.concat = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, token, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(token.unsqueeze(1)))
        output, hidden = self.lstm(embedded, hidden)
        
        attn_weights = torch.bmm(output, encoder_outputs.transpose(1, 2))
        attn_weights = F.softmax(attn_weights, dim=-1)
        context = torch.bmm(attn_weights, encoder_outputs)
        
        concat_input = torch.cat((output, context), dim=2)
        concat_output = torch.tanh(self.concat(concat_input))
        logits = self.fc(concat_output.squeeze(1))
        return logits, hidden

## Section 4: Complete Seq2Seq Model

Combines the Encoder and Luong Attention Decoder into one model.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, src_lens, tgt, teacher_forcing_ratio=0.5):
        batch_size, max_len = tgt.shape
        vocab_size = self.decoder.fc.out_features
        
        encoder_outputs, hidden = self.encoder(src, src_lens)
        
        outputs = torch.zeros(batch_size, max_len, vocab_size, device=DEVICE)
        token = tgt[:, 0]

        for t in range(1, max_len):
            logits, hidden = self.decoder(token, hidden, encoder_outputs)
            outputs[:, t] = logits
            teacher_force = random.random() < teacher_forcing_ratio
            token = tgt[:, t] if teacher_force else logits.argmax(dim=1)
        
        return outputs

## Section 5: Data Loading

Loads the pre-processed data generated by `01_preprocessing.ipynb`.

In [ ]:
PROCESSED_DIR = Path("../../data/processed")

with open(PROCESSED_DIR / "token2idx.json") as f:
    token2idx = json.load(f)
idx2token = {v: k for k, v in token2idx.items()}

with open(PROCESSED_DIR / "train.json") as f:
    train_df = pd.DataFrame(json.load(f))
with open(PROCESSED_DIR / "dev.json") as f:
    dev_df = pd.DataFrame(json.load(f))
with open(PROCESSED_DIR / "test.json") as f:
    test_df = pd.DataFrame(json.load(f))

class WikiQADataset(Dataset):
    def __init__(self, df):
        self.q_ids = df["q_ids"].tolist()
        self.a_ids = df["a_ids"].tolist()
    def __len__(self):
        return len(self.q_ids)
    def __getitem__(self, idx):
        return torch.tensor(self.q_ids[idx]), torch.tensor(self.a_ids[idx])

def collate_batch(batch):
    questions, answers = zip(*batch)
    src = pad_sequence(questions, batch_first=True, padding_value=PAD_IDX)
    tgt = pad_sequence(answers, batch_first=True, padding_value=PAD_IDX)
    return {"src": src, "tgt": tgt, "srcLens": torch.tensor([len(q) for q in questions])}

train_loader = DataLoader(WikiQADataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
dev_loader = DataLoader(WikiQADataset(dev_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(WikiQADataset(test_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

print(f"Data loaded! Train samples: {len(train_loader.dataset)}")

## Section 6: Training
## Training Helper Function (train_epoch)

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    """One full epoch of training."""
    model.train()
    total_loss = 0.0
    
    for batch in loader:
        src = batch["src"].to(DEVICE)
        tgt = batch["tgt"].to(DEVICE)
        src_lens = batch["srcLens"]
        
        optimizer.zero_grad()
        
        output = model(src, src_lens, tgt, TEACHER_FORCING_RATIO)
        
        loss = criterion(output[:, 1:].reshape(-1, output.size(-1)),
                        tgt[:, 1:].reshape(-1))
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

## Section 6.1: Training

Trains Model 2 using the same training loop style as Model 1.

In [ ]:
import time
import math

# =====================================================
# Training Configuration
# =====================================================
EPOCHS = 30
PATIENCE = 5

# 自動建立 models 資料夾（解決你而家嘅錯誤）
CHECKPOINT_PATH = Path("../../models/model2_luong_best.pt")
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Create Model 2
encoder = Encoder(len(token2idx), EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
decoder = LuongAttnDecoder(len(token2idx), EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
model2 = Seq2Seq(encoder, decoder).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model2.parameters(), lr=LEARNING_RATE)

trainLosses, valLosses = [], []
bestValLoss = float("inf")
epochsNoImprove = 0

print("🚀 Starting training of Model 2 (Luong Attention)...\n")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    trainLoss = train_epoch(model2, train_loader, optimizer, criterion)
    valLoss = train_epoch(model2, dev_loader, optimizer, criterion)
    
    elapsed = time.time() - t0
    
    trainLosses.append(trainLoss)
    valLosses.append(valLoss)

    marker = ""
    if valLoss < bestValLoss:
        bestValLoss = valLoss
        epochsNoImprove = 0
        torch.save({
            "model_state": model2.state_dict(),
            "token2idx": token2idx,
            "embed_dim": EMBED_DIM,
            "hidden_dim": HIDDEN_DIM,
            "n_layers": N_LAYERS,
            "dropout": DROPOUT,
        }, CHECKPOINT_PATH)
        marker = "  ✓ best model saved"
    else:
        epochsNoImprove += 1
        if epochsNoImprove >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
            break

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {trainLoss:.4f} (ppl {math.exp(trainLoss):6.2f}) | "
        f"val loss {valLoss:.4f} (ppl {math.exp(valLoss):6.2f}) | "
        f"{elapsed:.1f}s{marker}"
    )

print(f"\n✅ Training finished! Best val loss: {bestValLoss:.4f}")
print(f"Best model saved at: {CHECKPOINT_PATH}")

## Section 7: Evaluation (BLEU Score + Sample Outputs)
Calculates BLEU-1 score and shows sample predictions with comparison to real answers.

In [ ]:
# =====================================================
# SECTION 7: Evaluation with Beam Search
# =====================================================
# 使用 Beam Search 代替 Greedy，減少重複，生成更好答案。

@torch.no_grad()
def generate_response(model, question_ids, max_len=30, beam_size=5):
   
    model.eval()
    src = torch.tensor([question_ids], device=DEVICE)
    src_lens = torch.tensor([len(question_ids)], device=DEVICE)
    
    encoder_outputs, hidden = model.encoder(src, src_lens)
    
    # Initialize beams: (score, sequence, hidden)
    beams = [(0.0, [SOS_IDX], hidden)]
    
    for _ in range(max_len):
        new_beams = []
        for score, seq, h in beams:
            if seq[-1] == EOS_IDX:
                new_beams.append((score, seq, h))
                continue
            
            token = torch.tensor([seq[-1]], device=DEVICE)
            logits, new_h = model.decoder(token, h, encoder_outputs)
            
            log_probs = F.log_softmax(logits, dim=-1)
            top_log_probs, top_indices = log_probs.topk(beam_size)
            
            for k in range(beam_size):
                new_score = score + top_log_probs[0, k].item()
                new_seq = seq + [top_indices[0, k].item()]
                new_beams.append((new_score, new_seq, new_h))
        
        beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_size]
    

    best_seq = beams[0][1]
    return [idx2token.get(i, "<UNK>") for i in best_seq if i not in (SOS_IDX, EOS_IDX)]


def evaluate_model(model, test_loader, num_examples=20):
    model.eval()
    smoothie = SmoothingFunction().method4
    bleu_scores = []
    examples = []
    
    for i, batch in enumerate(test_loader):
        if i >= num_examples:
            break
        src = batch["src"].to(DEVICE)
        tgt = batch["tgt"].to(DEVICE)
        src_lens = batch["srcLens"]
        
        for j in range(min(3, src.size(0))):
            q_ids = src[j].cpu().tolist()
            pred_tokens = generate_response(model, q_ids)
            pred_text = " ".join(pred_tokens)
            
            q_text = " ".join([idx2token.get(idx, "<UNK>") for idx in q_ids if idx not in (PAD_IDX, SOS_IDX, EOS_IDX)])
            real_text = " ".join([idx2token.get(idx, "<UNK>") for idx in tgt[j].tolist() if idx not in (PAD_IDX, SOS_IDX, EOS_IDX)])
            
            bleu = sentence_bleu([real_text.split()], pred_text.split(), smoothing_function=smoothie)
            bleu_scores.append(bleu)
            
            examples.append({
                "Question": q_text,
                "Real Answer": real_text,
                "Model2 Prediction": pred_text,
                "BLEU": round(bleu, 4)
            })
    
    avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0
    print(f"\n✅ Model 2 Average BLEU-1 Score: {avg_bleu:.4f}\n")
    
    for ex in examples[:6]:
        print("Question :", ex["Question"])
        print("Real     :", ex["Real Answer"])
        print("Model2   :", ex["Model2 Prediction"])
        print("BLEU     :", ex["BLEU"])
        print("-" * 70)
    
    return examples, avg_bleu


# Run evaluation
samples, bleu_score = evaluate_model(model2, test_loader, num_examples=20)

# Run evaluation
samples, bleu_score = evaluate_model(model2, test_loader, num_examples=20)

## 8. Save Best Model 2

Save the trained Model 2 (with best validation loss) for later use and report.

In [ ]:
# Save the trained Model 2
CHECKPOINT_PATH = Path("../../models/model2_luong_best.pt")
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state": model2.state_dict(),
    "token2idx": token2idx,
    "embed_dim": EMBED_DIM,
    "hidden_dim": HIDDEN_DIM,
    "n_layers": N_LAYERS,
    "dropout": DROPOUT,
}, CHECKPOINT_PATH)

print(f"✅ Model 2 saved successfully at: {CHECKPOINT_PATH}")

## 9. Training Loss Curve

Visualise how training and validation loss changed over epochs.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(trainLosses, label='Training Loss', marker='o')
plt.plot(valLosses, label='Validation Loss', marker='o')
plt.title('Model 2 Training Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

## 10. Model 1 vs Model 2 Comparison

Side-by-side comparison on the same test samples (required for the project report and video).

In [ ]:
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ================== Model 1 Predictions  ==================
model1_predictions = [
    "the the the the the the the the the the the the the the the the the the the the",
    "the the the the the the the the the the the the the the the the the the the the",
    "the the the the the the the the the the the the the the the the the the the the",
    "the the the the the the the the the the the the the the the the the the the the",
    "the the the the the the the the the the the the the the the the the the the the",
    "the the the the the the the the the the the the the the the the the the the the",
]

# ================== Model 2 Results   ==================
model2_examples = samples

# ================== Calculate Model 1 BLEU ==================
smoothie = SmoothingFunction().method4
data = []

for i, ex in enumerate(model2_examples[:len(model1_predictions)]):
    model1_pred = model1_predictions[i]
    real_answer = ex["Real Answer"]
    
    # 計算 Model 1 BLEU
    model1_bleu = sentence_bleu([real_answer.split()], model1_pred.split(), smoothing_function=smoothie)
    
    data.append({
        "Question": ex["Question"],
        "Real Answer": real_answer,
        "Model 1 (No Attention)": model1_pred,
        "Model 2 (Luong Attention + Beam Search)": ex["Model2 Prediction"],
        "Model 1 BLEU": round(model1_bleu, 4),
        "Model 2 BLEU": ex["BLEU"]
    })

df = pd.DataFrame(data)
print("✅ Model 1 vs Model 2 Comparison Table (with BLEU)")
display(df)

# Optional: save to CSV for report
df.to_csv("../../results/model1_vs_model2_comparison.csv", index=False)
print("✅ Table saved as model1_vs_model2_comparison.csv")

✅ Model 1 vs Model 2 Comparison Table (with BLEU)


,Question,Real Answer,Model 1 (No Attention),Model 2 (Luong Attention + Beam Search),Model 1 BLEU,Model 2 BLEU
0,how african americans were immigrated to the us,as such african immigrants are to be distingui...,the the the the the the the the the the the th...,it is is the the in the in the in the the of t...,0.0094,0.0093
1,how a water pump works,pumps operate by some mechanism typically reci...,the the the the the the the the the the the th...,it is is a a a is a of is a american of and and a,0.0130,0.0122
2,how old was sue lyon when she made lolita,the actress who played lolita sue lyon was fou...,the the the the the the the the the the the th...,it was is a in in the in the in the and and an...,0.0155,0.0197
3,what does s h i e l d stand for,the acronym originally stood for supreme headq...,the the the the the the the the the the the th...,it is is a a the of of the the of the the and ...,0.0130,0.0149
4,what day is st patricks day,saint patrick 's day or the feast of saint pat...,the the the the the the the the the the the th...,it is is a a the of of the the of of the and the,0.0121,0.0301
5,how many books are included in the protestant ...,christian bibles range from the sixty six book...,the the the the the the the the the the the th...,it was the the the in the the in the the the t...,0.0151,0.0309


✅ Table saved as model1_vs_model2_comparison.csv


## Model 2: Seq2Seq LSTM with Luong Dot Attention

### 1. Model Architecture
Model 2 extends the basic seq2seq architecture of Model 1 by incorporating **Luong Dot Attention**. The encoder remains the same (single-layer LSTM), but the decoder now computes attention weights at every time step to focus on the most relevant parts of the input question. This allows the model to dynamically weigh different words in the question when generating each answer token.

### 2. Training Summary

| Epoch | Train Loss | Train PPL | Val Loss   | Val PPL   |
|-------|------------|-----------|------------|-----------|
| 1     | 8.3569     | 4259.59   | 7.3719     | 1590.73   |
| ...   | ...        | ...       | ...        | ...       |
| 30    | 5.6002     | 270.48    | **5.2622** | **192.91** |

- Training ran for the full 30 epochs (early stopping was not triggered).
- Both training and validation loss decreased steadily, showing that the model continued to learn.
- Final validation perplexity improved significantly compared to Model 1.

### 3. Evaluation Results (Test Set)

- **Average BLEU-1 Score**: **0.0201**
- Beam Search (beam size = 5) was used during inference to reduce repetition.

**Sample Outputs (selected examples)**

| Question                                      | Real Answer (shortened)                          | Model 1 Prediction                  | Model 2 Prediction (Luong + Beam)              | Model 1 BLEU | Model 2 BLEU |
|-----------------------------------------------|--------------------------------------------------|-------------------------------------|------------------------------------------------|--------------|--------------|
| how african americans were immigrated to the us | ... descendants of mostly west and central africans... | the the the the the the the the the the | it is is the the in the in the in the the of the | N/A          | 0.0093       |
| how a water pump works                        | pumps operate by some mechanism...               | the the the the the the the the the the | it is is a a a is a of is a american of and and a | N/A          | 0.0122       |
| how old was sue lyon when she made lolita     | ... sue lyon was fourteen...                     | the the the the the the the the the the | it was is a in in the in the in the and and and the | N/A          | 0.0197       |
| what does s h i e l d stand for               | ... supreme headquarters international espionage... | the the the the the the the the the the | it is is a a the of of the the of the the and the | N/A          | 0.0149       |

*(Full table with 20 samples is available in the attached CSV file: `model1_vs_model2_comparison.csv`)*

### 4. Comparison with Model 1 (No Attention)

- **BLEU-1 Score**: Model 2 (0.0201) is slightly higher than Model 1.
- **Output Quality**: Model 2 produces slightly more varied tokens than Model 1, but both models still suffer from heavy repetition and very short outputs.
- **Training Efficiency**: Model 2 shows better convergence (lower final val loss) thanks to the attention mechanism.
- **Limitation**: Despite the theoretical advantage of attention, the very small dataset (~728 training pairs) limits the model’s ability to learn meaningful question-answer mapping.

### 5. Interpretation and Limitations

The addition of Luong Attention improved the model’s ability to focus on relevant parts of the question, as evidenced by the lower validation loss. However, due to the small size of the WikiQA correct-answers-only dataset, both models remain in an early stage of learning and fail to generate fluent, content-rich answers. The outputs are dominated by high-frequency stop words (“the”, “is”, “a”, “in”), indicating that the models have not yet learned strong conditional generation.

This result is consistent with expectations for a small-scale seq2seq task trained from scratch. Larger datasets and more advanced techniques (e.g., pre-trained embeddings or transformer models) would be needed to achieve human-like performance.

